In [0]:
yrinvo_tbl = dbutils.widgets.get("yrinvo_tbl")
hchb_yrinvo_view = dbutils.widgets.get("hchb_yrinvo_view")
hchbofficemapping_tbl = dbutils.widgets.get("hchbofficemapping_tbl")
hchb_yrinvo_final_view = dbutils.widgets.get("hchb_yrinvo_final_view")
mart_client_tbl = dbutils.widgets.get("mart_client_tbl")
mart_client_hchb_view = dbutils.widgets.get("mart_client_hchb_view")
mart_client_cubhub_view = dbutils.widgets.get("mart_client_cubhub_view")
mart_client_bears_view = dbutils.widgets.get("mart_client_bears_view")
oblisthistory_tbl = dbutils.widgets.get("oblisthistory_tbl")
oblist_view = dbutils.widgets.get("oblist_view")
billing_fact_tbl = dbutils.widgets.get("billing_fact_tbl")
billing_reference_tbl = dbutils.widgets.get("billing_reference_tbl")
mart_billing_cte_view = dbutils.widgets.get("mart_billing_cte_view")
hchb_cleaned_view = dbutils.widgets.get("hchb_cleaned_view")
payerdimension_tbl = dbutils.widgets.get("payerdimension_tbl")
cubeserviceofficetxnsourcesystem_tbl = dbutils.widgets.get("cubeserviceofficetxnsourcesystem_tbl")
date_tbl = dbutils.widgets.get("date_tbl")
office_tbl = dbutils.widgets.get("office_tbl")
client_episode_fs_tbl = dbutils.widgets.get("client_episode_fs_tbl")
client_episodes_all_tbl = dbutils.widgets.get("client_episodes_all_tbl")
denialtype_tbl = dbutils.widgets.get("denialtype_tbl")

In [0]:

# Execute SQL using f-string and spark.sql
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW {hchb_yrinvo_view} AS
WITH yrinvo_cte AS (
    SELECT BranchID, InvNum, GroupID, CltId, PrimID
    FROM {yrinvo_tbl}
    WHERE BranchID <> 'COR'
)
SELECT *
FROM yrinvo_cte
""")

In [0]:

# Execute SQL using f-string and spark.sql
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW {hchb_yrinvo_final_view} AS
SELECT COALESCE(
        CASE 
            WHEN h1.BranchID RLIKE '[a-zA-Z]' THEN CAST(om.TargetOfficeNumber AS STRING) 
            ELSE NULL 
        END, 
        h1.BranchID
    ) AS BranchID,
    h1.InvNum,
    h1.GroupID,
    h1.CltId,
    h1.PrimID
FROM {hchb_yrinvo_view} h1
LEFT JOIN {hchbofficemapping_tbl} om
  ON h1.BranchID = om.SourceOfficeCode
""")

In [0]:

# Execute SQL using f-string and spark.sql
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW {mart_client_hchb_view} AS 
SELECT clientKey, SourceSystemId, OfficeNumber, rnb
FROM (
  SELECT clientKey, SourceSystemId, OfficeNumber, 
         ROW_NUMBER() OVER (PARTITION BY SourceSystemId, OfficeNumber ORDER BY SourceSystemId, OfficeNumber) AS rnb
  FROM {mart_client_tbl}
  WHERE OfficeNumber <> 0 AND SourceSystem = 'HCHB'
) a
WHERE rnb = 1
""")

In [0]:

# Execute SQL using f-string and spark.sql
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW {mart_client_cubhub_view} AS 
SELECT clientKey, ConformedLastName, ConformedFirstName, OfficeNumber, rnb
FROM (
  SELECT clientKey, ConformedLastName, ConformedFirstName, OfficeNumber, 
         ROW_NUMBER() OVER (PARTITION BY ConformedLastName, ConformedFirstName ORDER BY clientKey DESC) AS rnb
  FROM {mart_client_tbl}
  WHERE OfficeNumber <> 0 AND SourceSystem = 'CUBHUB'
) a
WHERE rnb = 1
""")

In [0]:

# Execute SQL using f-string and spark.sql
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW {mart_client_bears_view} AS
SELECT clientKey, SourceSystemId, OfficeNumber, rnb
FROM (
  SELECT clientKey, SourceSystemId, OfficeNumber, 
         ROW_NUMBER() OVER (PARTITION BY SourceSystemId, OfficeNumber ORDER BY SourceSystemId, OfficeNumber) AS rnb
  FROM {mart_client_tbl}
  WHERE OfficeNumber <> 0 AND SourceSystem = 'BEARS'
) d
WHERE rnb = 1
""")

In [0]:

# Execute SQL using f-string and spark.sql
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW {oblist_view} AS
SELECT office, invno, payorname, PAYORTYPE, BILLTO, CLIENTNO, rnb
FROM (
  SELECT office, invno, payorname, PAYORTYPE, BILLTO, CLIENTNO,
         ROW_NUMBER() OVER (PARTITION BY invno, OFFICE ORDER BY invno, OFFICE) as rnb
  FROM {oblisthistory_tbl}
  WHERE invno <> 'adv'
) a
WHERE rnb = 1
""")

In [0]:
# Execute SQL using f-string and spark.sql
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW {mart_billing_cte_view} AS
WITH max_fact_load_timestamp AS (
    SELECT COALESCE(MAX(Loaded_ts), CAST('1900-01-01 00:00:00' AS TIMESTAMP)) AS max_loaded_ts
    FROM {billing_fact_tbl}
)
SELECT m.*
FROM {billing_reference_tbl} m
CROSS JOIN max_fact_load_timestamp
WHERE m.Loaded_ts > max_fact_load_timestamp.max_loaded_ts
""")

In [0]:


spark.sql(f"""
INSERT INTO {billing_fact_tbl}
WITH {hchb_cleaned_view} AS (
  SELECT 
    hf.*,
    try_cast(CASE WHEN TRIM(hf.BranchID) = '' THEN NULL ELSE hf.BranchID END AS INT) AS BranchID_clean,
    CAST(hf.GroupID AS STRING) AS GroupID_clean,
    hf.PrimID AS PrimID_clean,
    hf.InvNum AS InvNum_clean
  FROM {hchb_yrinvo_final_view} AS hf
),
data_hchb AS (
  SELECT 
    cast(date_format(dt_svc_start.WeekEndingDate, 'yyyyMMdd') as int) AS Reporting_Week_Ending_Date_Key,
    dt_svc_start.DateKey AS Service_start_key,
    dt_svc_end.DateKey AS Service_end_key,
    s.SourceSystemKey AS Source_System_Key,
    ofc.OfficeKey AS OfficeKey,
    pd.PayerKey AS PayerKey,
    mh.clientKey AS ClientKey,
    mb.Adjudication_Date AS Adjudication_Date,
    mb.Check_EFT_Number AS Check_EFT_Number,
    mb.Grouping AS Grouping,
    mb.Import_Date AS Import_Date,
    mb.Member_ID AS Member_ID,
    mb.Patient_Ctl_No AS Patient_Ctl_No,
    mb.Patient_Name AS Patient_Name,
    mb.Payee_Name AS Payee_Name,
    mb.Payee_NPI AS Payee_NPI,
    mb.Payer_ICN AS Payer_ICN,
    mb.Payer_Name AS Payer_Name,
    mb.Payment_Date AS Payment_Date,
    mb.Received_Date AS Received_Date,
    mb.Subscriber_Name AS Subscriber_Name,
    mb.Note_Date AS Note_Date,
    mb.Notes AS Notes,
    mb.Appeal_Create_Date AS Appeal_Create_Date,
    mb.Appeal_Created AS Appeal_Created,
    mb.Assignment AS Assignment,
    mb.Followup AS Followup,
    mb.ID AS ID,
    mb.Patient_DOB AS Patient_DOB,
    mb.Review_Date AS Review_Date,
    mb.Reviewed AS Reviewed,
    mb.Reviewed_Reason AS Reviewed_Reason,
    mb.Capital_Outlier AS Capital_Outlier,
    mb.Operating_Outlier AS Operating_Outlier,
    mb.Reimbursement_Defined AS Reimbursement_Defined,
    CAST(mb.Status_Code AS DOUBLE) AS Status_Code,
    mb.Contract_Adj AS Contract_Adj,
    mb.Rendering_NPI AS Rendering_NPI,
    mb.Rendering_Provider AS Rendering_Provider,
    mb.Claim_Payment AS Claim_Payment,
    mb.Modifiers AS Modifiers,
    mb.Adjustment_Code AS Adjustment_Code,
    mb.Adjustment_Code_Description AS Adjustment_Code_Description,
    mb.Allowed_Amount AS Allowed_Amount,
    mb.Charged_Amount AS Charged_Amount,
    mb.Expected_Reimbursement AS Expected_Reimbursement,
    mb.HCPCS AS HCPCS,
    mb.Patient_Resp AS Patient_Resp,
    mb.Payment_Amount AS Payment_Amount,
    mb.Remit_Remark_Code AS Remit_Remark_Code,
    mb.Remit_Remark_Description AS Remit_Remark_Description,
    mb.Revenue_Code AS Revenue_Code,
    mb.Service_Line_Adjustments AS Service_Line_Adjustments,
    mb.Underpayment AS Underpayment,
    mb.Underpayment_Amount AS Underpayment_Amount,
    mb.Units AS Units,
    mb.Units_Paid AS Units_Paid,
    mb.service_line_no as service_line_no,
    dl.Denial_Type_Key AS Denial_Type_Key,
    current_timestamp as Loadedts
  FROM {mart_billing_cte_view} AS mb
  JOIN {hchb_cleaned_view} AS hf 
    ON TRIM(mb.Patient_Ctl_No) = TRIM(CAST(hf.InvNum_clean AS STRING)) AND (mb.Patient_Ctl_No RLIKE '^2[89][01239]' OR mb.Patient_Ctl_No LIKE '1%')
  LEFT JOIN {cubeserviceofficetxnsourcesystem_tbl} s
    ON s.SourceSystemName = 'HCHB'
  LEFT JOIN {date_tbl} dt_svc_start
    ON dt_svc_start.CalendarDate = mb.Service_Start
  LEFT JOIN {date_tbl} dt_svc_end
    ON dt_svc_end.CalendarDate = mb.Service_End
  LEFT JOIN {office_tbl} ofc
    ON ofc.OfficeNumber = hf.BranchID_clean
  LEFT JOIN {payerdimension_tbl} pd
    ON pd.PayerID = hf.GroupID_clean
  LEFT JOIN {client_episode_fs_tbl} cefs
    ON cefs.cefs_id = hf.PrimID_clean
  LEFT JOIN {client_episodes_all_tbl} cea
    ON cea.epi_id = cefs.cefs_epiid
  LEFT JOIN {mart_client_hchb_view} mh
    ON mh.SourceSystemId = CAST(cea.epi_id AS STRING)
       AND hf.BranchID_clean = mh.OfficeNumber
       AND mh.rnb = 1
  LEFT JOIN {denialtype_tbl} dl
    ON dl.Denial_Type = CASE 
                          WHEN mb.Service_Line_Adjustments > 0 THEN 'Adjustment'
                          WHEN mb.Service_Line_Adjustments < 0 THEN 'Reversal'
                          WHEN mb.Service_Line_Adjustments = 0 THEN 'Payment'
                        END
    
),

data_cubhub as(
  SELECT 
   cast(date_format(dt_svc_start.WeekEndingDate, 'yyyyMMdd') as int) AS Reporting_Week_Ending_Date_Key,
    dt_svc_start.DateKey AS Service_start_key,
    dt_svc_end.DateKey AS Service_end_key,
    s.SourceSystemKey AS Source_System_Key,
    ofc.OfficeKey AS OfficeKey,
    pd.PayerKey AS PayerKey,
    mc.clientKey AS ClientKey,
    mb.Adjudication_Date AS Adjudication_Date,
    mb.Check_EFT_Number AS Check_EFT_Number,
    mb.Grouping AS Grouping,
    mb.Import_Date AS Import_Date,
    mb.Member_ID AS Member_ID,
    mb.Patient_Ctl_No AS Patient_Ctl_No,
    mb.Patient_Name AS Patient_Name,
    mb.Payee_Name AS Payee_Name,
    mb.Payee_NPI AS Payee_NPI,
    mb.Payer_ICN AS Payer_ICN,
    mb.Payer_Name AS Payer_Name,
    mb.Payment_Date AS Payment_Date,
    mb.Received_Date AS Received_Date,
    mb.Subscriber_Name AS Subscriber_Name,
    mb.Note_Date AS Note_Date,
    mb.Notes AS Notes,
    mb.Appeal_Create_Date AS Appeal_Create_Date,
    mb.Appeal_Created AS Appeal_Created,
    mb.Assignment AS Assignment,
    mb.Followup AS Followup,
    mb.ID AS ID,
    mb.Patient_DOB AS Patient_DOB,
    mb.Review_Date AS Review_Date,
    mb.Reviewed AS Reviewed,
    mb.Reviewed_Reason AS Reviewed_Reason,
    mb.Capital_Outlier AS Capital_Outlier,
    mb.Operating_Outlier AS Operating_Outlier,
    mb.Reimbursement_Defined AS Reimbursement_Defined,
    CAST(mb.Status_Code AS DOUBLE) AS Status_Code,
    mb.Contract_Adj AS Contract_Adj,
    mb.Rendering_NPI AS Rendering_NPI,
    mb.Rendering_Provider AS Rendering_Provider,
    mb.Claim_Payment AS Claim_Payment,
    mb.Modifiers AS Modifiers,
    mb.Adjustment_Code AS Adjustment_Code,
    mb.Adjustment_Code_Description AS Adjustment_Code_Description,
    mb.Allowed_Amount AS Allowed_Amount,
    mb.Charged_Amount AS Charged_Amount,
    mb.Expected_Reimbursement AS Expected_Reimbursement,
    mb.HCPCS AS HCPCS,
    mb.Patient_Resp AS Patient_Resp,
    mb.Payment_Amount AS Payment_Amount,
    mb.Remit_Remark_Code AS Remit_Remark_Code,
    mb.Remit_Remark_Description AS Remit_Remark_Description,
    mb.Revenue_Code AS Revenue_Code,
    mb.Service_Line_Adjustments AS Service_Line_Adjustments,
    mb.Underpayment AS Underpayment,
    mb.Underpayment_Amount AS Underpayment_Amount,
    mb.Units AS Units,
    mb.Units_Paid AS Units_Paid,
    mb.service_line_no as service_line_no,
    dl.Denial_Type_Key AS Denial_Type_Key,
    current_timestamp as Loadedts

from {mart_billing_cte_view} as mb
LEFT JOIN (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY Name ORDER BY PayerKey) AS rnb
  FROM {payerdimension_tbl}
  WHERE SourceSystemKey = 19
) pd
  ON mb.Payer_Name = pd.Name AND pd.rnb = 1
LEFT JOIN {cubeserviceofficetxnsourcesystem_tbl} s
    ON s.SourceSystemName = 'CUBHUB'
LEFT JOIN {date_tbl} dt_svc_start
    ON dt_svc_start.CalendarDate = mb.Service_Start
LEFT JOIN {date_tbl} dt_svc_end
    ON dt_svc_end.CalendarDate = mb.Service_End
LEFT JOIN {mart_client_cubhub_view} mc
    ON TRIM(UPPER(mb.Patient_Name)) = TRIM(UPPER(concat(mc.ConformedFirstName,' ',mc.ConformedLastName)))
LEFT JOIN {office_tbl} ofc
    ON ofc.OfficeNumber = mc.OfficeNumber
LEFT JOIN {denialtype_tbl} dl
    ON dl.Denial_Type = CASE 
                          WHEN mb.Service_Line_Adjustments > 0 THEN 'Adjustment'
                          WHEN mb.Service_Line_Adjustments < 0 THEN 'Reversal'
                          WHEN mb.Service_Line_Adjustments = 0 THEN 'Payment'
                        END

    WHERE
      mb.patient_ctl_no RLIKE '^[0-9].*[0-9][EF][a-zA-Z][0-9].*[0-9]$'
   OR mb.patient_ctl_no RLIKE '^[0-9].*[0-9][EF][a-zA-Z]S[0-9].*[0-9]$'
   OR mb.patient_ctl_no RLIKE '^M-[0-9].*[0-9][EF][a-zA-Z][0-9].*[0-9]$'
   OR mb.patient_ctl_no RLIKE '^M-[0-9].*[0-9][EF][a-zA-Z]S[0-9].*[0-9]$'
),

data_bears AS (
  SELECT
  CAST(date_format(dt_svc_start.WeekEndingDate, 'yyyyMMdd') AS INT) AS Reporting_Week_Ending_Date_Key,
  dt_svc_start.DateKey AS Service_start_key,
  dt_svc_end.DateKey AS Service_end_key,
  CASE WHEN ob.INVNO IS NOT NULL THEN s.SourceSystemKey END AS Source_System_Key,
  ofc.OfficeKey,
  pd.PayerKey,
  mcb.ClientKey,
  mb.Adjudication_Date,
  mb.Check_EFT_Number,
  mb.Grouping,
  mb.Import_Date,
  mb.Member_ID,
  mb.Patient_Ctl_No,
  mb.Patient_Name,
  mb.Payee_Name,
  mb.Payee_NPI,
  mb.Payer_ICN,
  mb.Payer_Name,
  mb.Payment_Date,
  mb.Received_Date,
  mb.Subscriber_Name,
  mb.Note_Date,
  mb.Notes,
  mb.Appeal_Create_Date,
  mb.Appeal_Created,
  mb.Assignment,
  mb.Followup,
  mb.ID,
  mb.Patient_DOB,
  mb.Review_Date,
  mb.Reviewed,
  mb.Reviewed_Reason,
  mb.Capital_Outlier,
  mb.Operating_Outlier,
  mb.Reimbursement_Defined,
  CAST(mb.Status_Code AS DOUBLE) AS Status_Code,
  mb.Contract_Adj,
  mb.Rendering_NPI,
  mb.Rendering_Provider,
  mb.Claim_Payment,
  mb.Modifiers,
  mb.Adjustment_Code,
  mb.Adjustment_Code_Description,
  mb.Allowed_Amount,
  mb.Charged_Amount,
  mb.Expected_Reimbursement,
  mb.HCPCS,
  mb.Patient_Resp,
  mb.Payment_Amount,
  mb.Remit_Remark_Code,
  mb.Remit_Remark_Description,
  mb.Revenue_Code,
  mb.Service_Line_Adjustments,
  mb.Underpayment,
  mb.Underpayment_Amount,
  mb.Units,
  mb.Units_Paid,
  mb.service_line_no,
  dl.Denial_Type_Key,
  current_timestamp() AS Loadedts
FROM
(
  SELECT mb.*
  FROM {mart_billing_cte_view} mb
  LEFT JOIN {hchb_cleaned_view} hf
    ON TRIM(mb.Patient_Ctl_No) = TRIM(CAST(hf.InvNum AS STRING))
   AND (mb.Patient_Ctl_No RLIKE '^2[89][01239]' OR mb.Patient_Ctl_No LIKE '1%')
  WHERE hf.InvNum IS NULL
    AND mb.Patient_Ctl_No NOT IN (
      SELECT DISTINCT r.Patient_Ctl_No
      FROM {mart_billing_cte_view} r
      WHERE r.patient_ctl_no RLIKE '^[0-9].*[0-9][EF][a-zA-Z][0-9].*[0-9]$'
         OR r.patient_ctl_no RLIKE '^[0-9].*[0-9][EF][a-zA-Z]S[0-9].*[0-9]$'
         OR r.patient_ctl_no RLIKE '^M-[0-9].*[0-9][EF][a-zA-Z][0-9].*[0-9]$'
         OR r.patient_ctl_no RLIKE '^M-[0-9].*[0-9][EF][a-zA-Z]S[0-9].*[0-9]$'
    )
) mb
LEFT JOIN {oblist_view} ob
  ON ob.INVNO = RIGHT(mb.Patient_Ctl_No, 8)
 AND LENGTH(mb.Patient_Ctl_No) <> 7
LEFT JOIN {office_tbl} ofc
  ON ofc.OfficeNumber = ob.Office
LEFT JOIN {payerdimension_tbl} pd
  ON pd.PayerID = ob.BILLTO
LEFT JOIN {cubeserviceofficetxnsourcesystem_tbl} s
  ON s.SourceSystemName = 'BEARS'
LEFT JOIN {date_tbl} dt_svc_start
  ON dt_svc_start.CalendarDate = mb.Service_Start
LEFT JOIN {date_tbl} dt_svc_end
  ON dt_svc_end.CalendarDate = mb.Service_End
LEFT JOIN {denialtype_tbl} dl
  ON dl.Denial_Type = CASE
                       WHEN mb.Service_Line_Adjustments > 0 THEN 'Adjustment'
                       WHEN mb.Service_Line_Adjustments < 0 THEN 'Reversal'
                       WHEN mb.Service_Line_Adjustments = 0 THEN 'Payment'
                     END
LEFT JOIN {mart_client_bears_view} mcb
  ON mcb.SourceSystemId = ob.CLIENTNO
 AND mcb.OfficeNumber = ob.Office
)
 
SELECT * FROM data_hchb
UNION ALL
SELECT * FROM data_cubhub
UNION ALL
SELECT * FROM data_bears
""")